In [2]:
import pandas as pd
import numpy as np

PATH = "/Users/phamthanh/Documents/ESSCA | MSc Finance & Data Analyst/14. ESG Portfolio/v1.2/quant_master_db.csv"
raw = pd.read_csv(PATH)

print("Columns:", raw.columns.tolist())
print(raw.head(3))

def to_wide_returns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return wide daily returns: index=date (datetime), columns=tickers, values=daily return (float).
    Auto-detect wide vs long.
    """
    cols = [c.lower() for c in df.columns]

    # Case A: long format: has date + ticker + return
    if ("date" in cols) and (("ticker" in cols) or ("symbol" in cols)) and (("return" in cols) or ("ret" in cols) or ("daily_return" in cols)):
        date_col = df.columns[cols.index("date")]
        tic_col = df.columns[cols.index("ticker")] if "ticker" in cols else df.columns[cols.index("symbol")]
        ret_col = df.columns[cols.index("return")] if "return" in cols else (df.columns[cols.index("ret")] if "ret" in cols else df.columns[cols.index("daily_return")])
        d = df[[date_col, tic_col, ret_col]].copy()
        d[date_col] = pd.to_datetime(d[date_col])
        wide = d.pivot(index=date_col, columns=tic_col, values=ret_col).sort_index()
        return wide

    # Case B: wide format: first column date, others tickers
    # Heuristic: if one column looks like date and many numeric columns
    # find date-like column
    date_candidates = [c for c in df.columns if str(c).lower() in ["date", "datetime", "time"]]
    if date_candidates:
        date_col = date_candidates[0]
        wide = df.copy()
        wide[date_col] = pd.to_datetime(wide[date_col])
        wide = wide.set_index(date_col).sort_index()
        return wide

    raise ValueError("Không detect được format. Bạn gửi mình columns/preview để mình map đúng.")

R = to_wide_returns(raw)

# clean
R = R.replace([np.inf, -np.inf], np.nan)
# drop tickers quá thiếu data
min_obs = int(len(R)*0.6)
R = R.dropna(axis=1, thresh=min_obs)
# fill missing nhẹ (optional) – hoặc bạn có thể giữ NaN
R = R.fillna(0.0)

print("Wide returns shape:", R.shape)
print("Date range:", R.index.min(), "->", R.index.max())
print("Sample tickers:", list(R.columns[:10]))

Columns: ['Date', 'RIC', 'PriceClose']
         Date  RIC  PriceClose
0  2025-06-17  A.N      116.09
1  2025-06-18  A.N      115.52
2  2025-06-20  A.N      115.56
Wide returns shape: (161904, 2)
Date range: 2025-06-17 00:00:00 -> 2026-02-24 00:00:00
Sample tickers: ['RIC', 'PriceClose']


In [3]:
import numpy as np
import pandas as pd

# Convert toàn bộ về numeric, cái nào không convert được -> NaN
R_num = R.apply(pd.to_numeric, errors="coerce")

# drop các cột mà gần như toàn NaN (không phải returns)
min_obs = int(len(R_num) * 0.6)
R_num = R_num.dropna(axis=1, thresh=min_obs)

# fill missing nhẹ (hoặc bạn có thể giữ NaN và drop khi cần)
R_num = R_num.replace([np.inf, -np.inf], np.nan).fillna(0.0)

print("After cleaning:", R_num.shape)
print(R_num.dtypes.head())

def compute_features(R: pd.DataFrame, lookback_mom=63):
    lb = min(lookback_mom, len(R))
    mom = (1 + R.tail(lb)).prod() - 1
    vol = R.std(ddof=0) * np.sqrt(252)
    shp = (R.mean() / (R.std(ddof=0) + 1e-12)) * np.sqrt(252)
    return mom, vol, shp

def rank_universe(R: pd.DataFrame, lookback_mom=63, w_mom=0.6, w_lowvol=0.4):
    mom, vol, shp = compute_features(R, lookback_mom=lookback_mom)
    mom_rank = mom.rank(pct=True)
    lowvol_rank = (-vol).rank(pct=True)
    score = w_mom * mom_rank + w_lowvol * lowvol_rank

    out = pd.DataFrame({
        "mom_3m": mom,
        "vol_ann": vol,
        "sharpe_proxy": shp,
        "score": score
    }).sort_values("score", ascending=False)
    return out

ranking = rank_universe(R_num)
ranking

After cleaning: (161904, 1)
PriceClose    float64
dtype: object


,mom_3m,vol_ann,sharpe_proxy,score
PriceClose,2.608288e+113,5470.062783,5.894187,1.0


In [4]:
CURRENT = ["MSI","MO","O","MCD","PG","SO","GE","PVH","TRV","DG","APAM","USFD"]

def pick_core_candidates(ranking: pd.DataFrame, core_n=8):
    # core ưu tiên low vol + sharpe_proxy cao, ít cần mom cực mạnh
    temp = ranking.copy()
    # composite core score: 50% low vol, 50% sharpe proxy
    core_score = 0.5*((-temp["vol_ann"]).rank(pct=True)) + 0.5*(temp["sharpe_proxy"].rank(pct=True))
    temp["core_score"] = core_score
    return temp.sort_values("core_score", ascending=False).head(core_n).index.tolist()

def pick_satellite_candidates(ranking: pd.DataFrame, sat_n=4):
    return ranking.head(sat_n).index.tolist()

core_pool = pick_core_candidates(ranking, core_n=10)
sat_pool  = pick_satellite_candidates(ranking, sat_n=8)

core_pool[:10], sat_pool[:8]

(['PriceClose'], ['PriceClose'])